# Rail Profiles and Hunting Patterns

We compare the supplied rail shapes with conicity and empty-truck acceleration
hunting across seven rail conditions. The same 28 trucks appear under each
condition, so these are descriptive comparisons. Follow
[INSTALL.md](../INSTALL.md) for setup and inputs.

In [ ]:
# Shared input locations; see INSTALL.md for the Teams download layout.
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "project_paths.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the repository; see INSTALL.md.")
sys.path.insert(0, str(ROOT / "src"))
from project_paths import data_directory, find_project_root, merged_file
ROOT = find_project_root(ROOT)
DATA = data_directory(ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Rail profile files are stored locally in the data folder
rail_folder = DATA / "Profiles" / "Rail profiles"

sorted(path.name for path in rail_folder.iterdir())

In [ ]:
# Read each BAN file (header lines first, then a blank line, then X, Y, A, C points)
rails = {}

for path in sorted(rail_folder.glob("*.BAN")):
    points = []
    in_points = False

    for line in path.read_text().splitlines():
        line = line.strip()
        if not in_points and line.startswith("ColumnDef="):
            columns = line.split("=")[1].split(",")
        elif not in_points and line == "" and len(points) == 0:
            in_points = True
        elif in_points and line != "":
            points.append([float(value) for value in line.split()])

    rails[path.stem.replace("RProfile ", "")] = pd.DataFrame(points, columns=columns)

# Check that each profile loaded correctly
for name, points in rails.items():
    print(name, points.shape)

Profiles 1–4 have separate left and right BAN files; profiles 5–7
have one file each. A zero left/right radius range for a single-file profile
means only one supplied shape was available; it does not establish that its
two physical rails are identical.

In [ ]:
# Analysis 1 - Estimate crown radius by fitting a parabola within 20 mm of the top of the rail
rail_shape = []

for name, points in rails.items():
    top = points.loc[points["Y"].idxmax()]
    near_top = points[abs(points["X"] - top["X"]) < 20]
    a = np.polyfit(near_top["X"], near_top["Y"], 2)[0]

    within_1mm = points[points["Y"] > top["Y"] - 1]

    rail_shape.append({
        "Profile": name,
        "Rail_Profile": int(name[0]),
        "Crown_Radius": -1 / (2 * a),
        "Width_1mm": within_1mm["X"].max() - within_1mm["X"].min()
    })

rail_shape = pd.DataFrame(rail_shape)
rail_shape.round(1)

In [ ]:
# Analysis 2 - Average the left and right rails so there is one row per rail profile
rail_summary = rail_shape.groupby("Rail_Profile").agg(
    Crown_Radius=("Crown_Radius", "mean"),
    Crown_Radius_Diff=("Crown_Radius", lambda r: r.max() - r.min()),
    Width_1mm=("Width_1mm", "mean")
)

rail_summary.round(1)

In [ ]:
# Analysis 3 - Load the merged dataset and keep one row per car, truck, and rail profile
final_data = pd.read_csv(merged_file(DATA))

truck_data = final_data.groupby(["Car", "Truck", "Rail_Profile"], as_index=False).agg(
    Hunted=("Empty_Truck_Hunted", "first"),
    Critical_Speed=("Min Empty Truck from Acc HS [mph]", "first"),
    Eq_Con=("Min Eq Con 3mm", "first")
)

truck_data.shape

In [ ]:
# Analysis 4 - Compare rail shape with hunting rate, critical speed, and conicity
rail_results = truck_data.groupby("Rail_Profile").agg(
    Hunting_Rate=("Hunted", "mean"),
    Median_Speed=("Critical_Speed", "median"),
    Mean_Eq_Con=("Eq_Con", "mean")
)

rail_summary = rail_summary.join(rail_results)
rail_summary.sort_values("Crown_Radius").round(3)

In [ ]:
# Analysis 5 - Rank correlation between rail shape and results across the 7 rail profiles
rail_summary.corr(method="spearman").loc[
    ["Crown_Radius", "Crown_Radius_Diff", "Width_1mm"],
    ["Mean_Eq_Con", "Hunting_Rate", "Median_Speed"]
].round(2)

Larger fitted crown radii line up with lower mean conicity across
these seven profiles. Profiles 5 and 6 are among the flattest and have lower
hunting fractions; profile 4 has a relatively high hunting fraction despite
its larger radius. Profiles 1 and 4 have the largest supplied left/right radius
differences. Seven rail conditions are too few to separate rail shape from
other differences or establish a cause.

In [ ]:
# Analysis 6 - Summarize critical speed for each rail profile and check the 80 mph cap
print(truck_data.groupby("Rail_Profile")["Critical_Speed"].describe().round(1))
print()
print("At 80 mph:", (truck_data["Critical_Speed"] == 80).sum(), "of", len(truck_data))
print("Max speed:", truck_data["Critical_Speed"].max())

Seventy assessments are recorded at the 80 mph simulation limit.
Those values mean onset was not observed below that limit. Profile 6 combines
a relatively high median recorded speed with the lowest individual speed;
its wide spread matters as well as its median.

In [ ]:
# Analysis 7 - Check how much of the variation in critical speed comes from the rail profile vs the truck
truck_data["Truck_ID"] = truck_data["Car"].astype(str) + "-" + truck_data["Truck"].astype(str)

overall_mean = truck_data["Critical_Speed"].mean()
total = ((truck_data["Critical_Speed"] - overall_mean) ** 2).sum()

rail_means = truck_data.groupby("Rail_Profile")["Critical_Speed"].transform("mean")
truck_means = truck_data.groupby("Truck_ID")["Critical_Speed"].transform("mean")

print("Rail profile:", round(((rail_means - overall_mean) ** 2).sum() / total, 2))
print("Truck:", round(((truck_means - overall_mean) ** 2).sum() / total, 2))

Truck-group means account for about 53% of the squared variation
in the recorded, capped speeds; rail-group means account for about 4% when
calculated separately. This describes larger differences between trucks in
these data. It does not isolate wheel-profile effects from other truck/car
differences or account for the censoring at 80 mph.

In [ ]:
# Analysis 8 - Count how many of the seven rail profiles each truck hunted on
hunting_table = truck_data.pivot_table(
    index=["Car", "Truck"],
    columns="Rail_Profile",
    values="Hunted",
    aggfunc="first"
).astype(int)

hunting_table["Rails_Hunted"] = hunting_table.sum(axis=1)

hunting_table.sort_values("Rails_Hunted")

Nine trucks meet the acceleration hunting criterion under all
seven rail conditions and three under none. These groups may be useful for
describing wheel differences, but they are labels made from the observed
outcomes, not independently validated clusters.

In [ ]:
# Analysis 9 - Check the relationship between conicity and critical speed within each rail profile
truck_data.groupby("Rail_Profile")[["Eq_Con", "Critical_Speed"]].corr(method="spearman").xs(
    "Eq_Con", level=1
)["Critical_Speed"].round(2)

In [ ]:
# Analysis 10 - Compare hunting rates above and below a few conicity cutoffs
for cutoff in [0.15, 0.20, 0.25]:
    low = truck_data[truck_data["Eq_Con"] < cutoff]
    high = truck_data[truck_data["Eq_Con"] >= cutoff]
    print("Eq Con", cutoff, "| below:", round(low["Hunted"].mean(), 2), "| above:", round(high["Hunted"].mean(), 2))

Assessments below 0.20 conicity have a lower hunting fraction in
this dataset. Within-rail correlations with recorded speed vary: profile 3
has a clearer negative association, while profiles 5 and 6 have small positive
correlations. Conicity alone does not separate hunting from no hunting. The
80 mph cap and repeated trucks limit this descriptive comparison.

In [ ]:
# Visual for Analysis 1 - Plot all rail profiles, then zoom in on the top of the rail head
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, points in rails.items():
    axes[0].plot(points["X"], points["Y"], label=name)
    axes[1].plot(points["X"], points["Y"], label=name)

axes[0].set_title("Rail Profiles")
axes[1].set_title("Top of the Rail Head")
axes[1].set_ylim(-3, 0.5)

for ax in axes:
    ax.set_xlabel("X (mm)")
    ax.set_ylabel("Y (mm)")

axes[0].legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

**Profiles 5 and 6 look a lot flatter and wider across the top than the others. Profile 7 is somewhere in between.**

In [ ]:
# Visual for Analysis 4 and 5 - Flat vs rounded rail idea: rail shape, then conicity, then hunting
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Compare the roundest rail (1L) with the flattest rail (5), lined up at the top of the rail
for name, color in [("1L", "red"), ("5", "blue")]:
    points = rails[name]
    top = points.loc[points["Y"].idxmax()]
    x = points["X"] - top["X"]
    y = points["Y"] - top["Y"]

    radius = rail_shape.loc[rail_shape["Profile"] == name, "Crown_Radius"].iloc[0]
    label = "Rail " + name + " (radius " + str(round(radius)) + " mm)"
    axes[0].plot(x[abs(x) < 25], y[abs(x) < 25], color=color, linewidth=2, label=label)

axes[0].set_xlabel("Distance from Top of Rail (mm)")
axes[0].set_ylabel("Height Below Top (mm)")
axes[0].set_title("Rounded vs Flat Rail Head")
axes[0].legend()

# Crown radius vs conicity
axes[1].scatter(rail_summary["Crown_Radius"], rail_summary["Mean_Eq_Con"])
for rail, row in rail_summary.iterrows():
    axes[1].annotate(str(rail), (row["Crown_Radius"], row["Mean_Eq_Con"]))

axes[1].set_xlabel("Crown Radius (mm) - larger is flatter")
axes[1].set_ylabel("Mean Eq Con 3mm")
axes[1].set_title("Crown Radius and Mean Conicity")

# Crown radius vs hunting rate
axes[2].scatter(rail_summary["Crown_Radius"], rail_summary["Hunting_Rate"] * 100)
for rail, row in rail_summary.iterrows():
    axes[2].annotate(str(rail), (row["Crown_Radius"], row["Hunting_Rate"] * 100))

axes[2].set_xlabel("Crown Radius (mm) - larger is flatter")
axes[2].set_ylabel("Hunting Rate (%)")
axes[2].set_title("Crown Radius and Hunting Fraction")

plt.tight_layout()
plt.show()

In [ ]:
# Visual for Analysis 6 - Compare critical speed across the seven rail profiles
truck_data.boxplot(column="Critical_Speed", by="Rail_Profile")

plt.axhline(80, color="gray", linestyle="--", label="80 mph test limit")
plt.legend()
plt.xlabel("Rail Profile")
plt.ylabel("Critical Speed (mph)")
plt.title("Empty-Car Critical Speed by Rail Profile")
plt.suptitle("")
plt.show()

In [ ]:
# Visual for Analysis 8 - Number of trucks by how many rail profiles they hunted on
hunting_table["Rails_Hunted"].value_counts().sort_index().plot(kind="bar")

plt.xlabel("Number of Rail Profiles Hunted")
plt.ylabel("Number of Trucks")
plt.title("Hunting Frequency per Truck")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Visual for Analysis 9 and 10 - Conicity vs critical speed, colored by rail profile
for rail, group in truck_data.groupby("Rail_Profile"):
    plt.scatter(group["Eq_Con"], group["Critical_Speed"], label=rail, alpha=0.7)

plt.xlabel("Truck Eq Con 3mm")
plt.ylabel("Critical Speed (mph)")
plt.title("Conicity vs Critical Speed")
plt.legend(title="Rail Profile")
plt.show()